# GPT 시리즈: 언어모델의 스케일링 - 실습 코드 1: GPT-2로 Zero-shot / Few-shot 프롬프팅

- Tutorial ID: `expand-gpt-series`
- Tutorial: GPT 시리즈: 언어모델의 스케일링
- Section ID: `expand-gpt-series-code-1`
- Section: 실습 코드 1: GPT-2로 Zero-shot / Few-shot 프롬프팅


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: GPT-2로 Zero-shot / Few-shot 프롬프팅
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

model_name = "gpt2-medium"  # 355M 파라미터
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model.eval()

def generate(prompt, max_length=100, temperature=0.7):
        inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
                        temperature=temperature,
            do_sample=True,
                        top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ── Zero-shot (GPT-2 핵심 능력) ──
zero_shot_prompt = """Translate English to French:
Cheese =>"""
print("=== Zero-shot ===")
print(generate(zero_shot_prompt, max_length=20))

# ── One-shot (GPT-3 핵심 능력) ──
one_shot_prompt = """Translate English to French:
Cat => Chat
Dog =>"""
print("\n=== One-shot ===")
print(generate(one_shot_prompt, max_length=20))

# ── Few-shot (GPT-3 핵심 능력) ──
few_shot_prompt = """Translate English to French:
Cat => Chat
Dog => Chien
House => Maison
Book =>"""
print("\n=== Few-shot ===")
print(generate(few_shot_prompt, max_length=25))

# ── Few-shot으로 감정 분석 ──
sentiment_prompt = """Review: This movie was terrible and boring.
Sentiment: Negative

Review: I loved this film, it was amazing!
Sentiment: Positive

Review: The food was okay, nothing special.
Sentiment:"""
print("\n=== Few-shot Sentiment ===")
print(generate(sentiment_prompt, max_length=30))